In [6]:
import os
import urllib.request

base = "https://storage.data.gov.my/pricecatcher/"
files = ["lookup_item.parquet", "lookup_premise.parquet"] + [
    f"pricecatcher_2026-{m:02d}.parquet" for m in range(1, 7)
]

raw_dir = "../data/raw"
os.makedirs(raw_dir, exist_ok=True)

for f in files:
    path = os.path.join(raw_dir, f)
    if os.path.exists(path):
        print("already have", f)
    else:
        urllib.request.urlretrieve(base + f, path)
        print("downloaded", f)

already have lookup_item.parquet
already have lookup_premise.parquet
already have pricecatcher_2026-01.parquet
already have pricecatcher_2026-02.parquet
already have pricecatcher_2026-03.parquet
already have pricecatcher_2026-04.parquet
already have pricecatcher_2026-05.parquet
already have pricecatcher_2026-06.parquet


In [7]:
import pandas as pd

item = pd.read_parquet("../data/raw/lookup_item.parquet")
premise = pd.read_parquet("../data/raw/lookup_premise.parquet")

print(item.shape, premise.shape)
display(item.head())
display(premise.head())

(797, 5) (3929, 6)


,item_code,item,unit,item_group,item_category
0,-1,NaN,NaN,NaN,NaN
1,1,AYAM BERSIH - STANDARD,1kg,BARANGAN SEGAR,AYAM
2,2,AYAM BERSIH - SUPER,1kg,BARANGAN SEGAR,AYAM
3,3,AYAM HIDUP,1kg,BARANGAN SEGAR,AYAM
4,9,DAGING KAMBING BEBIRI IMPORT BERTULANG (MUTTON...,1kg,BARANGAN SEGAR,DAGING


,premise_code,premise,address,premise_type,state,district
0,-1,Unknown Premise,Unknown Address,NaN,NaN,NaN
1,2,PASAR BESAR IPOH,"JALAN LAKSAMANA,TAMAN JUBILEE,30300 IPOH, PERAK",Pasar Basah,Perak,Kinta
2,3,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
3,6,KEDAI RUNCIT TAFAZ MAJU PRESINT 9,"NO.5, JALAN P9B/1,PRESINT 9,62250 PUTRAJAYA",Kedai Runcit,W.P. Putrajaya,Wp Putrajaya
4,7,KEDAI RUNCIT SYAZ MAJU PRESINT 9,"NO.5, JALAN P9E/1,PRESINT 9,62250 PUTRAJAYA",Kedai Runcit,W.P. Putrajaya,Wp Putrajaya


In [8]:
prices = pd.read_parquet("../data/raw/pricecatcher_2026-01.parquet")

print(prices.shape)
print(prices.dtypes)
print(prices.isna().sum())
print(prices["date"].min(), prices["date"].max())
print(prices["price"].describe())
print("duplicate rows:", prices.duplicated(["date", "premise_code", "item_code"]).sum())

(1976483, 4)
date             object
premise_code      int64
item_code         int64
price           float64
dtype: object
date            0
premise_code    0
item_code       0
price           0
dtype: int64
2026-01-01 2026-01-31
count    1.976483e+06
mean     1.252541e+01
std      1.091877e+01
min      3.000000e-02
25%      5.400000e+00
50%      9.600000e+00
75%      1.528000e+01
max      1.418000e+03
Name: price, dtype: float64
duplicate rows: 0


In [9]:
import os
print(sorted(os.listdir("../data/raw")))

print(item["item_group"].value_counts(dropna=False))
print(item["item_category"].value_counts(dropna=False).head(30))
print(premise["premise_type"].value_counts(dropna=False))
print(sorted(premise["state"].dropna().unique()))

['lookup_item.parquet', 'lookup_premise.parquet', 'pricecatcher_2026-01.parquet', 'pricecatcher_2026-02.parquet', 'pricecatcher_2026-03.parquet', 'pricecatcher_2026-04.parquet', 'pricecatcher_2026-05.parquet', 'pricecatcher_2026-06.parquet']
item_group
BARANGAN BERBUNGKUS         219
MAKANAN SIAP MASAK          191
BARANGAN SEGAR              164
MINUMAN                      50
SUSU DAN BARANGAN BAYI       50
BARANGAN KEDAI SERBANEKA     49
PRODUK KEBERSIHAN            46
BARANGAN KERING              27
NaN                           1
Name: count, dtype: int64
item_category
LAUK                               72
MINUMAN                            47
SAYUR-SAYURAN                      41
BAHAN LAUT                         40
MINYAK DAN LEMAK                   39
DAGING                             32
REMPAH RATUS (BERBUNGKUS)          30
MEE / BIHUN / KUEY TEOW            28
SUSU BAYI                          27
LAIN-LAIN                          27
TERSEDIA MINUM                     26
B

In [10]:
print("items not in lookup:", (~prices["item_code"].isin(item["item_code"])).sum())
print("premises not in lookup:", (~prices["premise_code"].isin(premise["premise_code"])).sum())
print("item_code -1 rows:", (prices["item_code"] == -1).sum())
print("premise_code -1 rows:", (prices["premise_code"] == -1).sum())

print(prices.groupby("date").size())

med = prices.groupby("item_code")["price"].transform("median")
ratio = prices["price"] / med
print("price > 5x item median:", (ratio > 5).sum())
print("price < 0.2x item median:", (ratio < 0.2).sum())
print(prices.assign(ratio=ratio).sort_values("ratio", ascending=False).head(10))

items not in lookup: 31242
premises not in lookup: 0
item_code -1 rows: 0
premise_code -1 rows: 0
date
2026-01-01     10757
2026-01-02     38294
2026-01-03     38318
2026-01-04     39038
2026-01-05    211381
2026-01-06     37851
2026-01-07     38264
2026-01-08     38385
2026-01-09     38688
2026-01-10     38471
2026-01-11     38719
2026-01-12    189192
2026-01-13     37700
2026-01-14     36909
2026-01-15     38728
2026-01-16     39149
2026-01-17     32971
2026-01-18     37730
2026-01-19    192538
2026-01-20     38790
2026-01-21     39092
2026-01-22     39079
2026-01-23     39196
2026-01-24     39217
2026-01-25     39542
2026-01-26    214559
2026-01-27     38466
2026-01-28     39219
2026-01-29    197554
2026-01-30     39469
2026-01-31     39217
dtype: int64
price > 5x item median: 40
price < 0.2x item median: 298
               date  premise_code  item_code   price       ratio
1325785  2026-01-23         21557        109   600.0  150.000000
1325801  2026-01-23         21557       2086  